In [1]:
import pandas as pd
import sqlite3
from datetime import datetime as dt
import os
from pipe.sales import principal
from data import entry

In [29]:
path = r'C:\Users\ramon\OneDrive\Desktop\estudos\Data_analysis\Sales_pipe\data\entry'

def load_csv(file):
    return pd.read_csv(os.path.join(path, file), encoding="latin1")

orders = load_csv("olist_orders_dataset.csv")
items = load_csv("olist_order_items_dataset.csv")
payments = load_csv("olist_order_payments_dataset.csv")
customers = load_csv("olist_customers_dataset.csv")
products = load_csv("olist_products_dataset.csv")

conn = sqlite3.connect(':memory:')

customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB


In [28]:
# Tratamento payments
payments_tratado = payments.groupby('order_id').agg({
    'payment_value' : 'sum',
    'payment_type' : 'first',
    'payment_sequential' : 'max',
    'payment_installments' : 'max'}).reset_index()
payments_tratado.head().sort_values(by = 'payment_installments', ascending = True)

,order_id,payment_value,payment_type,payment_sequential,payment_installments
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,credit_card,1,2
3,00024acbcdf0a6daa1e931b038114c75,25.78,credit_card,1,2
1,00018f77f2f0320c557190d7a144bdd3,259.83,credit_card,1,3
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,credit_card,1,3
2,000229ec398224ef6ca0657da4fc703e,216.87,credit_card,1,5


In [ ]:
#Interação entre Order e Items pra criação da coluna de diff_shipping_limit
items ['shipping_limit_date'] = pd.to_datetime(order['shipping_limit_date'])
merge_order_items = ['order_id', 'order_delivered_carrier_date']
items = pd.merge(items, orders[colunas_interest], on='order_id', how='left')
items['shipping_acurracy'] = items['shipping_limit_date'] - items['order_delivered_carrier_date']
items = items[['order_id','order_item_id','product_id','seller_id','shipping_limit_date','price','freight_value','shipping_acurracy']]


In [ ]:
# Tratamento das colunas de order
cols_datas = ['order_purchase_timestamp', 'order_approved_at', 
              'order_delivered_carrier_date', 'order_delivered_customer_date', 
              'order_estimated_delivery_date']

for col in cols_datas:
    order[col] = pd.to_datetime(order[col])

orders['approve_intervall'] = orders['order_purchase_timestamp'] - orders['order_approved_at']
orders['deliver_intervall'] = orders['order_delivered_carrier_date'] - orders['order_delivered_customer_date']
orders['estimated_delivery'] = orders['order_approved_at'] - orders['order_estimated_delivery_date']

orders = orders[['approve_intervall','deliver_intervall','estimated_delivery','order_id','customer_id']]

In [ ]:
# Criando os SQLs para o futuro merge
orders.to_sql('pedidos', conn, index=False, if_exists='replace')
items.to_sql('itens',conn,index=False,if_exists ='replace')
payments.to_sql('pagamentos',conn,index=False,if_exists ='replace')
customers.to_sql('clientes',conn,index=False,if_exists ='replace')
products.to_sql('produtos',conn,index=False,if_exists ='replace')


# Sobre o projeto

## Notebook
Este notebook vai me servir como início para as análises que entrarão na minha SalesPipe. Vou identificar os problemas deste dataframe e realizar a tratativa das funções, a partir disso começo a desenvolver as funções que vão entrar na pipe.

Idealizei este modelo pois ficaria um tanto trabalhoso de mais ficar imaginando por conta própria quais seriam as necessidades de um caso real.

## Sobre o DataFrame
Ao extrair os dados recebi 5 dataframes diferentees, creio eu deveriam ser alocados em um banco de dados relacional, funcionam commo um DataLake simples.
Como o foco do projeto não é sobre modelagem de Banco de dados ou algo do tipo, toda a tratativa será realizada por meio de merges na pipe principal do projeto.

Mas o foco deste projeto é em habilidades de Engenharia, ETL e Analise de Dados.

## Informações

Como o meu dataframe final me retorna algumas informações como:
- Preço do produto
- Valor da venda (Produto + Frete)
- Tempo esperado de entrega
- Tempo real de entrega
- Status da ordem
- Valor de Frete
- Ano e Mês da venda

Tenho bastante material para trabalhar para montar as KPI´s

### KPI´s

- CMV -> Custo médio por venda (como só tenho o frete seria o Valor médio do Frete)
- Ticket Médio de Venda
- Tempo médio de entrega
- Faturamento histórico
- Tendência Histórica
- OTD -> On time delivery

Dentro das informações deste dataframe estas informações já geram um panorama bastante produtivo.

Como dentro do dataframe só tenho os IDs tanto de produtos quanto de 